# Proyecto: Clasificación de sentimientos en reseñas de películas
Autor: Adrián Robles Arques

En este proyecto vamos a proceder a implementar dos modelos de análisis de sentimientos, el primero es el modelo de Naive Bayes, y el segundo es el modelo de clasificación de sentimentos basado en redes neuronales.

Para esto, vamos a emplear la librería NLTK, que nos permitirá extraer las palabras clave de las reseñas, y también para clasificarlas con el modelo de Naive Bayes. Emplearemos el conjunto de datos de reseñas de películas que se encuentra en el repositorio de GitHub de NLTK, que se llama movie_reviews. El conjunto de datos contiene 2000 reseñas de películas de 1937, 1938, 1939, 1940, 1941 y 1942. Cada reseña está clasificada como positiva o negativa. 

## FASE 1: Cargar y preprocesar el dataset

En esta fase vamos a cargar el dataset movie_reviews de NLTK y realizar un preprocesamiento estándar: conversión a minúsculas, eliminación de puntuación, eliminación de stopwords y tokenización.

In [1]:
# Importamos las librerías necesarias
import nltk
import string
from nltk.tokenize import ToktokTokenizer
import re
nltk.download('movie_reviews')
nltk.download('stopwords')
from nltk.corpus import movie_reviews

[nltk_data] Downloading package movie_reviews to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
# Cargamos el dataset de movie_reviews
reviews = [(movie_reviews.raw(fileid), category) 
            for category in movie_reviews.categories() 
            for fileid in movie_reviews.fileids(category)]

# Definimos una función para limpiar el texto
def limpiar_texto(texto):
    
    texto = re.sub(r'\W', ' ', str(texto))
    texto = re.sub(r'\s+[a-zA-Z]\s+', ' ', texto)
    texto = re.sub(r'\s+', ' ', texto, flags=re.I)
    texto = texto.lower()
    
    return texto

reviews_cleaned = []

for review, category in reviews:
    # Limpiamos el texto de la reseña
    review_cleaned = limpiar_texto(review)
    
    reviews_cleaned.append((category, review_cleaned))

## FASE 2: Modelos de clasificaición

En esta segunda fase vamos a implementar dos modelos de clasificación: Naive Bayes y Redes Neuronales. Para su entrenamiento usaremos los datos procesados en la primera fase, dividiendo el dataset en un conjunto de entrenamiento y otro de validación.

In [7]:
# Importamos las librerías necesarias para el modelo

import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, classification_report

# Importamos las librerías de Keras y Tensorflow
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam

import os
tf.compat.v1.enable_eager_execution()
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [8]:
# Separamos textos y etiquetas
texts = []
labels = []

for category, text in reviews_cleaned:
    texts.append(text)
    # Convertimos las etiquetas a números (pos=1, neg=-1)
    labels.append(1 if category == 'pos' else -1)

print(f'Total de reseñas: {len(texts)}')

# Dividimos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f'Entrenamiento: {len(X_train)} reseñas')
print(f'Prueba: {len(X_test)} reseñas')

Total de reseñas: 2000
Entrenamiento: 1600 reseñas
Prueba: 400 reseñas


In [9]:
# MODELO 1: NAIVE BAYES
print('=== MODELO NAIVE BAYES ===')

# Vectorización con TF-IDF
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Entrenamiento del modelo Naive Bayes
nb_model = MultinomialNB(alpha=1.0)
nb_model.fit(X_train_tfidf, y_train)

# Predicciones
y_pred_nb = nb_model.predict(X_test_tfidf)

# Evaluación
accuracy_nb = accuracy_score(y_test, y_pred_nb)
f1_nb = f1_score(y_test, y_pred_nb)

print(f'Accuracy Naive Bayes: {accuracy_nb:.4f}')
print(f'F1-Score Naive Bayes: {f1_nb:.4f}')
print('\nInforme de clasificación Naive Bayes:')
print(classification_report(y_test, y_pred_nb, target_names=['Negativo', 'Positivo']))

=== MODELO NAIVE BAYES ===
Accuracy Naive Bayes: 0.8375
F1-Score Naive Bayes: 0.8363

Informe de clasificación Naive Bayes:
              precision    recall  f1-score   support

    Negativo       0.83      0.84      0.84       200
    Positivo       0.84      0.83      0.84       200

    accuracy                           0.84       400
   macro avg       0.84      0.84      0.84       400
weighted avg       0.84      0.84      0.84       400



In [19]:
# MODELO 2: RED NEURONAL LSTM
print('\n=== MODELO LSTM ===')

# Configuración de parámetros
max_words = 10000
max_length = 200
embedding_dim = 128

# Tokenización para LSTM
tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

# Convertir textos a secuencias
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

# Padding de secuencias
X_train_pad = pad_sequences(X_train_seq, maxlen=max_length, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_length, padding='post', truncating='post')

# Convertir etiquetas a arrays numpy
y_train_binary = np.array([(1 if y == 1 else 0) for y in y_train])
y_test_binary = np.array([(1 if y == 1 else 0) for y in y_test])

print(f'Forma de X_train_pad: {X_train_pad.shape}')
print(f'Forma de y_train: {y_train_binary.shape}')


=== MODELO LSTM ===
Forma de X_train_pad: (1600, 200)
Forma de y_train: (1600,)


In [29]:
# Construcción del modelo LSTM
lstm_model = Sequential(name='LSTM_Model')

# Embedding layer
lstm_model.add(Embedding(
    input_dim=max_words,
    output_dim=embedding_dim,
    trainable=True
))

# LSTM layer - configuración más estable
lstm_model.add(LSTM(
    units=64,
    return_sequences=False,
    dropout=0.6,          # Dropout interno del LSTM
    recurrent_dropout=0.3 # Dropout recurrente
))

# Capas densas
lstm_model.add(Dense(32, activation='relu'))
lstm_model.add(Dropout(0.3))
lstm_model.add(Dense(16, activation='relu'))
lstm_model.add(Dropout(0.3))
lstm_model.add(Dense(1, activation='sigmoid'))

# Crear el modelo
print("Creando modelo LSTM...")


# Compilar con optimizador específico
lstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Resumen del modelo LSTM:")
lstm_model.summary()

Creando modelo LSTM...
Resumen del modelo LSTM:


Model: "LSTM_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_9 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_9 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Entrenamiento del modelo LSTM
print("Entrenando modelo LSTM...")

history = lstm_model.fit(
    X_train_pad, y_train_binary,
    epochs=10,
    batch_size=256,
    validation_split=0.2,
    verbose=1
)

# Evaluación del modelo LSTM
print("\nEvaluando modelo LSTM...")
loss, accuracy_lstm = lstm_model.evaluate(X_test_pad, y_test_binary, verbose=0)

# Predicciones LSTM
y_pred_lstm_prob = lstm_model.predict(X_test_pad, verbose=0)
y_pred_lstm = (y_pred_lstm_prob > 0.5).astype(int).flatten()

# Métricas LSTM
f1_lstm = f1_score(y_test_binary, y_pred_lstm)

print(f'Accuracy LSTM: {accuracy_lstm:.4f}')
print(f'F1-Score LSTM: {f1_lstm:.4f}')
print('\nReporte de clasificación LSTM:')
print(classification_report(y_test_binary, y_pred_lstm, target_names=['Negativo', 'Positivo']))

Entrenando modelo LSTM...
Epoch 1/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 3s 247ms/step - accuracy: 0.5193 - loss: 0.6929 - val_accuracy: 0.4563 - val_loss: 0.6934
Epoch 2/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 164ms/step - accuracy: 0.5440 - loss: 0.6924 - val_accuracy: 0.4594 - val_loss: 0.6936
Epoch 3/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 169ms/step - accuracy: 0.5349 - loss: 0.6921 - val_accuracy: 0.4563 - val_loss: 0.6937
Epoch 4/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 169ms/step - accuracy: 0.5277 - loss: 0.6919 - val_accuracy: 0.4531 - val_loss: 0.6936
Epoch 5/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 165ms/step - accuracy: 0.5819 - loss: 0.6893 - val_accuracy: 0.4750 - val_loss: 0.6929
Epoch 6/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 166ms/step - accuracy: 0.6243 - loss: 0.6860 - val_accuracy: 0.5281 - val_loss: 0.6914
Epoch 7/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 164ms/step - accuracy: 0.6622 - loss: 0.6787 - val_accuracy: 0.5437 - val_loss: 0.6885
Epoch 8/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 166ms/step - accuracy: 0.7413 - loss: 0.6599 - val_ac

In [ ]:
# Comparación de modelos
import matplotlib.pyplot as plt

print("\n=== COMPARACIÓN DE MODELOS ===")
print(f"Naive Bayes - Accuracy: {accuracy_nb:.4f}, F1-Score: {f1_nb:.4f}")
print(f"LSTM        - Accuracy: {accuracy_lstm:.4f}, F1-Score: {f1_lstm:.4f}")

# Gráfico de comparación
models = ['Naive Bayes', 'LSTM']
accuracies = [accuracy_nb, accuracy_lstm]
f1_scores = [f1_nb, f1_lstm]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico de Accuracy
ax1.bar(models, accuracies, color=['skyblue', 'lightcoral'], alpha=0.7)
ax1.set_title('Comparación de Accuracy')
ax1.set_ylabel('Accuracy')
ax1.set_ylim(0, 1)
for i, v in enumerate(accuracies):
    ax1.text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom')

# Gráfico de F1-Score
ax2.bar(models, f1_scores, color=['lightgreen', 'orange'], alpha=0.7)
ax2.set_title('Comparación de F1-Score')
ax2.set_ylabel('F1-Score')
ax2.set_ylim(0, 1)
for i, v in enumerate(f1_scores):
    ax2.text(i, v + 0.01, f'{v:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

In [ ]:
# Función para probar el modelo con nuevas reseñas
def predecir_sentimiento(texto, modelo='ambos'):
    """
    Predice el sentimiento de un texto dado
    """
    # Preprocesar el texto
    texto_limpio = limpiar_texto(texto)
    tokens = ToktokTokenizer().tokenize(texto_limpio)
    tokens = [token for token in tokens if token not in stopwords and len(token) > 1]
    texto_procesado = ' '.join(tokens)
    
    resultados = {}
    
    if modelo in ['nb', 'ambos']:
        # Predicción con Naive Bayes
        texto_tfidf = vectorizer.transform([texto_procesado])
        pred_nb = nb_model.predict_proba(texto_tfidf)[0]
        resultados['Naive Bayes'] = {
            'prediccion': 'Positivo' if pred_nb[1] > 0.5 else 'Negativo',
            'confianza': max(pred_nb)
        }
    
    if modelo in ['lstm', 'ambos']:
        # Predicción con LSTM
        texto_seq = tokenizer.texts_to_sequences([texto_procesado])
        texto_pad = pad_sequences(texto_seq, maxlen=max_length, padding='post')
        pred_lstm = lstm_model.predict(texto_pad, verbose=0)[0][0]
        resultados['LSTM'] = {
            'prediccion': 'Positivo' if pred_lstm > 0.5 else 'Negativo',
            'confianza': pred_lstm if pred_lstm > 0.5 else 1 - pred_lstm
        }
    
    return resultados

# Ejemplo de uso
texto_ejemplo = "This movie was absolutely fantastic! Great acting and amazing plot."
print("=== PRUEBA CON TEXTO NUEVO ===")
print(f"Texto: {texto_ejemplo}")
print("\nPredicciones:")
predicciones = predecir_sentimiento(texto_ejemplo)
for modelo, resultado in predicciones.items():
    print(f"{modelo}: {resultado['prediccion']} (confianza: {resultado['confianza']:.3f})")